# Assistant réglementaire Solvency II / IFRS 17 : préparation des données et recherche

Pipeline : PDF → texte → nettoyage → chunks → embeddings + BM25 → recherche (références exactes + hybride).

Exécutez les cellules **de haut en bas**. Pour repartir de zéro : Restart, puis Exécuter tout.

## 1. Configuration

In [1]:
import os, re, json
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\yolde\RAG\rag-regulatory")
if not Path("data/raw").exists() and PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)

# Une seule fois si besoin (sinon laisser en commentaire) :
# %pip install pymupdf sentence-transformers

print("Dossier de travail :", Path.cwd())
print("PDF :", [p.name for p in Path("data/raw").glob("*.pdf")])

Dossier de travail : c:\Users\yolde\RAG\rag-regulatory
PDF : ['ifrs17_eu_2021_2036.pdf', 'solvency2_directive_2009_138.pdf']


## 2. Extraction du texte des PDF
Ne fait rien si les `.txt` existent déjà.

In [2]:
RAW, TEXT = Path("data/raw"), Path("data/text")
TEXT.mkdir(parents=True, exist_ok=True)

todo = [p for p in RAW.glob("*.pdf") if not (TEXT / (p.stem + ".txt")).exists()]
if todo:
    import pymupdf
    for pdf_path in todo:
        doc = pymupdf.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc)
        (TEXT / (pdf_path.stem + ".txt")).write_text(text, encoding="utf-8")
        print(f"{pdf_path.name} : {len(doc)} pages, {len(text)} caractères")
else:
    print("Textes déjà extraits, rien à faire.")

Textes déjà extraits, rien à faire.


## 3. Chargement des textes

In [3]:
solv = Path("data/text/solvency2_directive_2009_138.txt").read_text(encoding="utf-8")
ifrs = Path("data/text/ifrs17_eu_2021_2036.txt").read_text(encoding="utf-8")
print(len(solv.split("\n")), "lignes (directive),", len(ifrs.split("\n")), "lignes (IFRS 17)")

18519 lignes (directive), 4213 lignes (IFRS 17)


## 4. Nettoyage
Suppression : espaces insécables, traits d'union invisibles, lignes parasites de chaque page, et **en-tête de page imprimé à l'envers dans la marge** (suite de caractères isolés `N`, `E`, `9`, `0`, ...).

In [4]:
NOISE = [
    re.compile(r"^EN$"),
    re.compile(r"^Official Journal of the European Union$"),
    re.compile(r"^L \d+/\d+$"),
    re.compile(r"^\d{1,2}\.\d{1,2}\.\d{4}$"),
]
MARGIN_TOKEN = re.compile(r"^(?:[0-9/LNE]|\d\.\d)$")

def drop_margin_text(lines, max_run=14):
    out, i = [], 0
    while i < len(lines):
        if lines[i] == "N" and i + 1 < len(lines) and lines[i + 1] == "E":
            j = i + 2
            while j < len(lines) and j - i <= max_run and MARGIN_TOKEN.match(lines[j]):
                j += 1
            i = j
        else:
            out.append(lines[i])
            i += 1
    return out

def is_short_noise(l):
    return len(l) <= 3 and not re.fullmatch(r"\d+\.|\(?[a-z0-9]+\)", l)

def drop_rotated_header(lines):
    """Supprime l'en-tête pivoté des pages en paysage (contient '2.2' et '1.7')."""
    out, i = [], 0
    while i < len(lines):
        if is_short_noise(lines[i]):
            j = i
            while j < len(lines) and is_short_noise(lines[j]):
                j += 1
            run = lines[i:j]
            if "2.2" in run and "1.7" in run:
                i = j
                continue
            out.extend(run)
            i = j
        else:
            out.append(lines[i])
            i += 1
    return out

def clean_lines(text):
    text = text.replace("\xa0", " ").replace("\xad\n", "").replace("\xad", "")
    out = []
    for l in text.split("\n"):
        l = l.strip()
        if not l or any(p.match(l) for p in NOISE):
            continue
        out.append(l)
    return drop_rotated_header(drop_margin_text(out))

## 5. Directive Solvency II : un chunk par article

In [5]:
def pack_paragraphs(lines, max_chars=3500):
    groups, cur = [], []
    for l in lines:
        if re.fullmatch(r"\d+\.", l) and cur:
            groups.append(cur)
            cur = []
        cur.append(l)
    if cur:
        groups.append(cur)
    parts, buf = [], ""
    for g in groups:
        t = " ".join(g)
        if buf and len(buf) + len(t) > max_chars:
            parts.append(buf)
            buf = t
        else:
            buf = (buf + " " + t).strip()
    if buf:
        parts.append(buf)
    return parts

sl = clean_lines(solv)

cand = [(i, int(l.split()[1])) for i, l in enumerate(sl) if re.fullmatch(r"Article \d+", l)]
headings, expected = [], 1
for i, n in cand:
    if n == expected:
        headings.append((i, n))
        expected += 1

last = headings[-1][0]
end_all = next((i for i in range(last, len(sl)) if sl[i] == "ANNEX I"), len(sl))

solv_chunks = []
for k, (start, n) in enumerate(headings):
    end = headings[k + 1][0] if k + 1 < len(headings) else end_all
    title = sl[start + 1]
    parts = pack_paragraphs(sl[start + 2 : end])
    for j, p in enumerate(parts, 1):
        solv_chunks.append({
            "id": f"SII-Art{n}" + (f"-p{j}" if len(parts) > 1 else ""),
            "source": "Directive 2009/138/EC (Solvency II)",
            "article": n,
            "title": title,
            "text": p,
        })

assert len({c["id"] for c in solv_chunks}) == len(solv_chunks), "identifiants en double"
print("Articles retrouvés :", len(headings), "(dernier :", headings[-1][1], ")")
print("Chunks directive :", len(solv_chunks))

Articles retrouvés : 312 (dernier : 312 )
Chunks directive : 332


## 6. IFRS 17 : un chunk par paragraphe
Corps (`1.`, `2.`...), Appendix A (définitions regroupées), Appendix B et C (`B1`, `C1`... sans point). Appendix D exclu.

In [6]:
il = clean_lines(ifrs)
il = il[next(i for i, l in enumerate(il) if l == "OBJECTIVE"):]

def find_line(lines, text, start=0):
    return next(i for i in range(start, len(lines)) if lines[i] == text)

iA = find_line(il, "Appendix A")
iB = find_line(il, "Appendix B", iA)
iC = find_line(il, "Appendix C", iB)
iD = find_line(il, "Appendix D", iC)
print("Appendices (lignes) :", iA, iB, iC, iD)

PARA = re.compile(r"^(?:(?:B|C)\d+[A-Z]*|\d+[A-Z]+\.?|\d+\.)$")
SOURCE = "IFRS 17 Insurance Contracts (Regulation (EU) 2021/2036)"

def is_heading(line):
    return (line.isupper() and len(line) >= 8 and len(line.split()) <= 12
            and not PARA.match(line) and not line.startswith("("))

def parse_numbered(lines, part):
    out, section, cur_id, cur_txt = [], part, None, []
    def flush():
        if cur_id and cur_txt:
            out.append({"paragraph": cur_id, "part": part,
                        "section": section, "text": " ".join(cur_txt)})
    for line in lines:
        if PARA.match(line):
            flush()
            cur_id, cur_txt = line.rstrip("."), []
        elif is_heading(line):
            section = line
        else:
            cur_txt.append(line)
    flush()
    return out

def parse_glossary(lines, part, max_chars=2500):
    out, buf = [], ""
    for line in lines:
        if buf and len(buf) + len(line) > max_chars:
            out.append(buf)
            buf = line
        else:
            buf = (buf + " " + line).strip()
    if buf:
        out.append(buf)
    return [{"paragraph": f"AppA-{k}", "part": part, "section": "Defined terms",
             "text": t} for k, t in enumerate(out, 1)]

raw = (parse_numbered(il[:iA], "Body")
       + parse_glossary(il[iA + 2 : iB], "Appendix A")
       + parse_numbered(il[iB:iC], "Appendix B")
       + parse_numbered(il[iC:iD], "Appendix C"))

ifrs_chunks, seen = [], {}
for c in raw:
    base = f"IFRS17-{c['paragraph']}"
    seen[base] = seen.get(base, 0) + 1
    cid = base if seen[base] == 1 else f"{base}-dup{seen[base]}"
    ifrs_chunks.append({"id": cid, "source": SOURCE, **c})

print("Chunks IFRS 17 :", len(ifrs_chunks))
print("Doublons :", [c["id"] for c in ifrs_chunks if "-dup" in c["id"]])

Appendices (lignes) : 1181 1317 2704 3085
Chunks IFRS 17 : 334
Doublons : []


## 7. Chunks finaux : découpe des chunks trop longs (> 2000 caractères), suppression des paragraphes vides ou supprimés, sauvegarde

In [7]:
def split_long(text, max_chars=2000):
    if len(text) <= max_chars:
        return [text]
    sents = re.split(r"(?<=[.;:])\s+", text)
    pieces = []
    for s in sents:
        while len(s) > max_chars:
            cut = s.rfind(" ", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            pieces.append(s[:cut])
            s = s[cut:].strip()
        pieces.append(s)
    parts, buf = [], ""
    for s in pieces:
        if buf and len(buf) + len(s) + 1 > max_chars:
            parts.append(buf)
            buf = s
        else:
            buf = (buf + " " + s).strip()
    if buf:
        parts.append(buf)
    return parts

def finalize(chunks, max_chars=2000, min_chars=30):
    out = []
    for c in chunks:
        t = c["text"].strip()
        if len(t) < min_chars or "[deleted]" in t.lower():
            continue
        parts = split_long(t, max_chars)
        for j, p in enumerate(parts, 1):
            d = dict(c)
            d["text"] = p
            if len(parts) > 1:
                d["id"] = f"{c['id']}-s{j}"
            out.append(d)
    return out

solv_final = finalize(solv_chunks)
ifrs_final = finalize(ifrs_chunks)

for name, ch in [("directive", solv_final), ("IFRS 17", ifrs_final)]:
    assert len({c["id"] for c in ch}) == len(ch), "identifiants en double"
    sizes = sorted(len(c["text"]) for c in ch)
    print(f"{name}: {len(ch)} chunks | min/médiane/max : {sizes[0]} / {sizes[len(sizes)//2]} / {sizes[-1]}")

Path("data/chunks").mkdir(parents=True, exist_ok=True)
for name, ch in [("solvency2", solv_final), ("ifrs17", ifrs_final)]:
    with open(f"data/chunks/{name}_final.jsonl", "w", encoding="utf-8") as f:
        for c in ch:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("Sauvegardé : data/chunks/*_final.jsonl")

directive: 407 chunks | min/médiane/max : 129 / 955 / 1999
IFRS 17: 348 chunks | min/médiane/max : 66 / 531 / 1991
Sauvegardé : data/chunks/*_final.jsonl


## 8. Contrôles de qualité
Après le nettoyage, il ne doit rester presque aucune séquence de caractères isolés, et l'Article 301 ne doit plus contenir `N E 9 0 0`.

In [8]:
def suspicious_runs(lines, min_len=3):
    runs, cur = [], []
    for i, l in enumerate(lines + [""] * 5):
        short = (len(l) <= 3 and l != ""
                 and not re.fullmatch(r"\d+\.", l)
                 and not re.fullmatch(r"\(?[a-z0-9]+\)", l))
        if short:
            cur.append((i, l))
        else:
            if len(cur) >= min_len:
                runs.append(cur)
            cur = []
    return runs

for name, lines in [("directive", sl), ("IFRS 17", il)]:
    runs = suspicious_runs(lines)
    print(f"{name} : {len(runs)} séquences suspectes restantes")
    for r in runs[:6]:
        print("  ligne", r[0][0], [l for _, l in r])

print("\nArt. 301 :", next(c for c in solv_final if c["article"] == 301)["text"][:250])
print("\nArt. 101 :", next(c for c in solv_final if c["article"] == 101)["text"][:300])
for pid in ["32", "B66"]:
    p = next((c for c in ifrs_final if c["paragraph"] == pid), None)
    print(f"\n§{pid} |", p["part"], "|", p["section"], "|", p["text"][:250] if p else "introuvable")

directive : 251 séquences suspectes restantes
  ligne 9777 ['0,5', '0', '0', '1']
  ligne 9790 ['O', 'jR', 'C', 'S', '×', 'iR', 'C', 'S', '×']
  ligne 9800 ['o', 'C', 'j,i', '∑', '√', '=']
  ligne 9807 ['n', 'o', 'n', 'R', 'C', 'S', 'j', 'jR', 'C', 'S', '×', 'iR', 'C', 'S', '×']
  ligne 9823 ['o', 'C', 'j,i', '∑', '√', '=', 'R', 'C', 'S']
  ligne 9869 ['jR', 'C', 'S', '×', 'iR', 'C', 'S', '×']
IFRS 17 : 0 séquences suspectes restantes

Art. 301 : 1. The Commission shall be assisted by the European Insurance and Occupational Pensions Committee. 2. Where reference is made to this paragraph, Articles 5 and 7 of Decision 1999/468/EC shall apply, having regard to the provisions of Article 8 thereo

Art. 101 : 1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursue its business as a going concern. 3. The Solvency Capital Requirement shall be calibr

## 9. Embeddings
Modèle `multilingual-e5-base` (512 tokens, multilingue). Les embeddings sont recalculés automatiquement si les chunks ont changé.

In [9]:
import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer

chunks = []
for name in ["solvency2_final", "ifrs17_final"]:
    with open(f"data/chunks/{name}.jsonl", encoding="utf-8") as f:
        chunks += [json.loads(l) for l in f]
print("Chunks chargés :", len(chunks))

def label(c):
    if "article" in c:
        return f"{c['source']} - Article {c['article']} - {c['title']}"
    return f"{c['source']} - paragraph {c['paragraph']} - {c['section']}"

model = SentenceTransformer("intfloat/multilingual-e5-base")
model.max_seq_length = 512

texts = ["passage: " + label(c) + "\n" + c["text"] for c in chunks]
sig = hashlib.md5("\n".join(texts).encode("utf-8")).hexdigest()
emb_path, sig_path = Path("data/chunks/embeddings.npy"), Path("data/chunks/embeddings.sig")

if emb_path.exists() and sig_path.exists() and sig_path.read_text() == sig:
    emb = np.load(emb_path)
    print("Embeddings relus depuis le disque (chunks inchangés).")
else:
    emb = model.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True)
    np.save(emb_path, emb)
    sig_path.write_text(sig)
    print("Embeddings recalculés et sauvegardés.")
print("Matrice d'embeddings :", emb.shape)

Chunks chargés : 755


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings relus depuis le disque (chunks inchangés).
Matrice d'embeddings : (755, 768)


## 10. Recherche par mots exacts (BM25)

In [10]:
import math
from collections import Counter

def tok(s):
    s = re.sub(r"(\d),(\d)", r"\1.\2", s.lower())   # 99,5 -> 99.5
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", s)

docs_tok = [tok(label(c) + " " + c["text"]) for c in chunks]
N = len(docs_tok)
avgdl = sum(len(d) for d in docs_tok) / N
df = Counter(t for d in docs_tok for t in set(d))
idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
tfs = [Counter(d) for d in docs_tok]

def bm25_scores(question, k1=1.5, b=0.75):
    out = np.zeros(N)
    for i, (tf, d) in enumerate(zip(tfs, docs_tok)):
        s = 0.0
        for t in tok(question):
            if t in tf:
                f = tf[t]
                s += idf[t] * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        out[i] = s
    return out

print("BM25 prêt :", N, "chunks")

BM25 prêt : 755 chunks


## 11. Recherche finale : références exactes + hybride (embeddings + BM25)

In [11]:
by_art, by_par = {}, {}
for i, c in enumerate(chunks):
    if "article" in c:
        by_art.setdefault(c["article"], []).append(i)
    else:
        by_par.setdefault(c["paragraph"].upper(), []).append(i)

def find_references(question):
    idx = []
    for m in re.finditer(r"\b(?:article|art\.?)\s*(\d+)", question, re.I):
        idx += by_art.get(int(m.group(1)), [])
    for m in re.finditer(r"(?:paragraphe?|para\.?|§)\s*([BC]?\d+[A-Z]?)\b", question, re.I):
        idx += by_par.get(m.group(1).upper(), [])
    for m in re.finditer(r"\b([BC]\d+[A-Z]?)\b", question):
        idx += by_par.get(m.group(1), [])
    return list(dict.fromkeys(idx))

def retrieve(question, k=5, pool=30, c=60):
    q = model.encode(["query: " + question], normalize_embeddings=True)[0]
    dense, sparse = emb @ q, bm25_scores(question)
    top_d = list(np.argsort(dense)[::-1][:pool])
    top_s = list(np.argsort(sparse)[::-1][:pool])
    fused = {}
    for rank, i in enumerate(top_d):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)
    for rank, i in enumerate(top_s):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)
    refs = find_references(question)
    ranked = refs + [i for i in sorted(fused, key=fused.get, reverse=True) if i not in refs]
    return ranked[:k], set(refs)

def show(question, k=5):
    ids, refs = retrieve(question, k)
    for i in ids:
        ch = chunks[i]
        ref = f"Art. {ch['article']}" if "article" in ch else f"IFRS 17 §{ch['paragraph']}"
        tag = "RÉFÉRENCE EXACTE" if i in refs else "recherche"
        print(f"{ch['id']} | {ref} | {tag}")
        print("   ", ch["text"][:200].replace("\n", " "), "...\n")

In [12]:
import bisect
starts = [i for i, n in headings]
nums = [n for i, n in headings]

by_article = Counter()
for r in suspicious_runs(sl):
    k = bisect.bisect_right(starts, r[0][0]) - 1
    by_article[nums[k] if k >= 0 else 0] += 1

print("Séquences restantes :", sum(by_article.values()))
print("Articles concernés  :", len(by_article))
print("Top 10 :", by_article.most_common(10))

Séquences restantes : 251
Articles concernés  : 1
Top 10 : [(312, 251)]


## 12. Tests

In [13]:
for q in [
    "What is the Solvency Capital Requirement and what confidence level does it use?",
    "Qu'est-ce que la marge de service contractuelle ?",
    "risk adjustment for non-financial risk",
    "Article 101",
    "IFRS 17 paragraph B66",
]:
    print("QUESTION :", q)
    show(q)
    print("=" * 60)

QUESTION : What is the Solvency Capital Requirement and what confidence level does it use?
SII-Art101 | Art. 101 | recherche
    1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursu ...

SII-Art122 | Art. 122 | recherche
    1. Insurance and reinsurance undertakings may use a different time period or risk measure than that set out in Article 101(3) for internal modelling purposes as long as the outputs of the internal mod ...

SII-Art104-s1 | Art. 104 | recherche
    1. The Basic Solvency Capital Requirement shall comprise individual risk modules, which are aggregated in accordance with point (1) of Annex IV. It shall consist of at least the following risk modules ...

SII-Art230 | Art. 230 | recherche
    consolidation-based method 1. The calculation of the group solvency of the participating insurance or reinsurance undertaking shall be c